In [ ]:
# CNN1 vs. CNN1+CNN2

In [ ]:
import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from keras.callbacks import EarlyStopping

# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')

def build_cnn_model(hp):
    model = models.Sequential()
    
    # Add a Conv1D layer with hyperparameters
    model.add(layers.Conv1D(
        filters=hp.Int('filters', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size', min_value=2, max_value=7, step=1),  # Kernel size smaller than or equal to input length
        activation='relu',
        padding='same',
        input_shape=(143, 4)  # Input shape (sequence_length, features)
    ))
    
    # Add a max pooling layer
    model.add(layers.MaxPooling1D(pool_size=2))  # Pooling layer (can reduce sequence length)
    
    # Flatten the output before passing to dense layers
    model.add(layers.Flatten())
    
    # Add dropout exactly before the Dense layer (dropout rate: 0 to 0.5)
    model.add(layers.Dropout(rate=hp.Float('dropout_rate', min_value=0.0, max_value=0.5)))
    
    # Add a Dense layer (fully connected layer)
    model.add(layers.Dense(
        hp.Int('dense_units', min_value=16, max_value=256, step=16),
        activation='relu'
    ))
    
    # Output layer for regression task (predicting a single value)
    model.add(layers.Dense(1))  # For regression, we need a single output unit
    
    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model
    


# Open the file and read its contents
with open('Feature_CNN1.txt', 'r') as file:
    data = []  # List to hold the 2D arrays
    current_array = []  # Temporary list for the current array

    for line in file:
        line = line.strip()  # Remove any leading/trailing whitespace

        # Check for a blank line
        if line == "":
            if current_array:  # If current_array has data, save it to data
                data.append(current_array)
                current_array = []  # Reset for the next block
        else:
            # Remove the brackets and split the line into elements
            line = line.replace('[', '').replace(']', '')  # Remove square brackets
            # Split the line by spaces, remove commas, and convert to floats
            # Strip any trailing commas or extra spaces from each number before converting to float
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])

    # Append the last array if it's not empty
    if current_array:
        data.append(current_array)
        
# Convert to numpy array
sample_data = np.array(data)
print(len(sample_data))

# Reshape for CNN input
X = sample_data.reshape(len(sample_data), 143, 4)  # (samples, timesteps, features)

rate = []
with open('HT3_indel_frequency_value_percentage.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)

# Define random seed to ensure input data consistency
np.random.seed(42)
# Generate 2000 random indices from the range [0, len(X))
indices_to_select = np.random.choice(len(X), size=1000, replace=False)
# Get all indices
all_indices = np.arange(len(X))
# Get the indices of the unselected data
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

# Select elements from X and y based on the random indices
X_selected = X[indices_to_select]
y_selected = y[indices_to_select]

# Get unselected elements as new data to test the model
X_unselected = X[indices_unselected]
y_unselected = y[indices_unselected]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_selected, test_size=0.2, random_state=42)


# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1.h5')

# Evaluate the final model on the test set
val_loss, val_mae = final_model.evaluate(X_test, y_test)
print(f"Final Best Model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict the values on the test set
y_pred = final_model.predict(X_test)
# Compute Spearman correlation between true values and predicted values
spearman_corr_test, _ = spearmanr(y_test, y_pred)
print(f"Final Best Model - Spearman Correlation for the test set: {spearman_corr_test}")

print()

# Predict the values on the new data set, 100 independent trials using the final best model

spearman_array = []

with open("outlier_feature_index_CNN2.txt", "r") as f:
    outlier_feature_index = [int(line.strip()) for line in f]


# Generate 1000 random indices from the unselected data
indices_to_select_new = np.random.choice(outlier_feature_index, size=len(outlier_feature_index), replace=False)
# Select elements from X and y based on the random indices
X_unselected_selected = X[indices_to_select_new]
y_unselected_selected = y[indices_to_select_new]

# Use the final saved model for prediction
y_pred = final_model.predict(X_unselected_selected)
for i in range (0, len(y_pred)):
    print(i)
    print(y_pred[i][0])
    print(y_unselected_selected[i])
    print(abs(y_pred[i][0]-y_unselected_selected[i]))
    print()

with open("12_abs_error_CNN1_CNN2_not_included.txt", "w") as f:
    for i in range(len(y_pred)):
        abs_err = abs(y_pred[i][0] - y_unselected_selected[i])
        f.write(f"{abs_err}\n")

with open("12_predicted_CNN1_CNN2_not_included.txt", "w") as f:
    for i in range(len(y_pred)):
        f.write(f"{y_pred[i][0]}\n")

with open("12_actual_CNN1_CNN2_not_included.txt", "w") as f:
    for i in range(len(y_unselected_selected)):
        f.write(f"{y_unselected_selected[i]}\n")

y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    
# Compute Spearman correlation between true values and predicted values
pearson_corr_new, _ = pearsonr(y_unselected_selected, y_pred)
print(pearson_corr_new)


In [ ]:
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping


# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')
                

def build_fusion_model(hp):
    """
    Builds a two-branch CNN model using the Keras Functional API.
    Each branch processes one input, and then the outputs are concatenated
    for a final prediction (regression).
    """

    # -------------------------------------------------------------------------
    # Branch 1 for input of shape (143, 4)
    # -------------------------------------------------------------------------
    input1 = keras.Input(shape=(143, 4), name='branch1_input')

    # Conv1D layer for branch 1
    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)


    # -------------------------------------------------------------------------
    # Branch 2 for input of shape (178, 3)
    # -------------------------------------------------------------------------
    input2 = keras.Input(shape=(178, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)

    # -------------------------------------------------------------------------
    # Fuse (concatenate) the outputs of both branches
    # -------------------------------------------------------------------------
    merged = layers.Concatenate()([x1, x2])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    # Optionally, you can add more dense layers after concatenation
    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)

    # -------------------------------------------------------------------------
    # Build and compile the final model
    # -------------------------------------------------------------------------
    model = keras.Model(inputs=[input1, input2], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model


# -----------------------------------------------------------------------------
# Data Loading and Preprocessing
# -----------------------------------------------------------------------------
# 1) Load first input data (shape = (n_samples, 143, 4))
#    The following is analogous to your original code for the base + PAM input.
with open('Feature_CNN1.txt', 'r') as file:
    data_branch1 = []
    current_array = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array:
                data_branch1.append(current_array)
                current_array = []
        else:
            line = line.replace('[', '').replace(']', '')
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array:
        data_branch1.append(current_array)


X_branch1 = np.array(data_branch1)  # shape: (n_samples, 143, 4)
X_branch1 = X_branch1.reshape(len(X_branch1), 143, 4)  # (samples, timesteps, features)

# 2) Load second input data (shape = (n_samples, 178, 3))
#    Adjust this part to match how your second input file is formatted.
#    For example, if you have something like "Feature_branch2.txt" with dimension (84,3).
with open('Feature_CNN2.txt', 'r') as file:
    data_branch2 = []
    current_array2 = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array2:
                data_branch2.append(current_array2)
                current_array2 = []

        else:
            line = line.replace('[', '').replace(']', '')
            current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array2:
        data_branch2.append(current_array2)

X_branch2 = np.array(data_branch2)  # shape: (n_samples, 178, 3)
X_branch2 = X_branch2.reshape(len(X_branch2), 178, 3)  # (samples, timesteps, features)

# 3) Load regression targets (y)
#    Make sure this matches the number of samples for both inputs.
rate = []
with open('HT3_indel_frequency_value_percentage.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)

# -----------------------------------------------------------------------------
# Ensure both X_branch1 and X_branch2 have the same first dimension as y
# -----------------------------------------------------------------------------
assert len(X_branch1) == len(X_branch2) == len(y), "All inputs must have the same number of samples."


# -----------------------------------------------------------------------------
# (Optional) Randomly select or filter data if you want
# -----------------------------------------------------------------------------
# For illustration, let's assume you want to select the same 1000 random samples
# from both X_branch1 and X_branch2. Then create a hold-out set from the rest.
np.random.seed(42)
indices_to_select = np.random.choice(len(X_branch1), size=1000, replace=False)
all_indices = np.arange(len(X_branch1))
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

X1_selected = X_branch1[indices_to_select]
X2_selected = X_branch2[indices_to_select]
y_selected = y[indices_to_select]

X1_unselected = X_branch1[indices_unselected]
X2_unselected = X_branch2[indices_unselected]
y_unselected = y[indices_unselected]


# -----------------------------------------------------------------------------
# Split the selected data into training and testing sets
# -----------------------------------------------------------------------------
X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1_selected, 
    X2_selected, 
    y_selected,
    test_size=0.2,
    random_state=42
)


# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1_CNN2.h5')

# -----------------------------------------------------------------------------
# Retrieve the Best Model and Evaluate
# -----------------------------------------------------------------------------
val_loss, val_mae = final_model.evaluate([X1_test, X2_test], y_test)
print(f"Best Fusion CNN model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict and compute Spearman correlation on the test set
y_pred_test = final_model.predict([X1_test, X2_test])
spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
print(f"Spearman Correlation (test set): {spearman_corr_test}")

print()


# -----------------------------------------------------------------------------
# Evaluate on new (unselected) data multiple times if desired
# -----------------------------------------------------------------------------
spearman_array = []

with open("outlier_feature_index_CNN2.txt", "r") as f:
    outlier_feature_index = [int(line.strip()) for line in f]

# Randomly select 1000 samples from the unselected portion
indices_to_select_new = np.random.choice(outlier_feature_index, size=len(outlier_feature_index), replace=False)

X1_unselected_selected = X_branch1[indices_to_select_new]
X2_unselected_selected = X_branch2[indices_to_select_new]
y_unselected_selected = y[indices_to_select_new]


# Predict on the new data
y_pred_new = final_model.predict([X1_unselected_selected, X2_unselected_selected])
for i in range (0, len(y_pred_new)):
    print(i)
    print(y_pred_new[i][0])
    print(y_unselected_selected[i])
    print(abs(y_pred_new[i][0]-y_unselected_selected[i]))
    print()
    
with open("12_abs_error_CNN1_CNN2_included.txt", "w") as f:
    for i in range(len(y_pred_new)):
        abs_err = abs(y_pred_new[i][0] - y_unselected_selected[i])
        f.write(f"{abs_err}\n")

with open("12_predicted_CNN1_CNN2_included.txt", "w") as f:
    for i in range(len(y_pred_new)):
        f.write(f"{y_pred_new[i][0]}\n")

y_pred_new = np.asarray(y_pred_new, dtype=np.float64).ravel()

# Compute Spearman correlation
pearson_corr_new, _ = pearsonr(y_unselected_selected, y_pred_new)
print(pearson_corr_new)



In [ ]:
# Calculate difference between CNN1 and CNN1+CNN2 predicted

In [ ]:
file1 = "12_abs_error_CNN1_CNN2_not_included.txt"
file2 = "12_abs_error_CNN1_CNN2_included.txt"
output = "difference_12_abs_error_CNN2_included.txt"

i = 0

with open(file1) as f1, open(file2) as f2, open(output, "w") as fout:
    for line1, line2 in zip(f1, f2):
        print(i)
        v1 = float(line1.strip())
        v2 = float(line2.strip())
        diff = v1 - v2
        print(diff)
        fout.write(f"{diff}\n")
        i += 1


In [ ]:
# CNN1+CNN2 vs. CNN1+CNN2+MLP

In [ ]:
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping


# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')
                

def build_fusion_model(hp):
    """
    Builds a two-branch CNN model using the Keras Functional API.
    Each branch processes one input, and then the outputs are concatenated
    for a final prediction (regression).
    """

    # -------------------------------------------------------------------------
    # Branch 1 for input of shape (143, 4)
    # -------------------------------------------------------------------------
    input1 = keras.Input(shape=(143, 4), name='branch1_input')

    # Conv1D layer for branch 1
    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)


    # -------------------------------------------------------------------------
    # Branch 2 for input of shape (178, 3)
    # -------------------------------------------------------------------------
    input2 = keras.Input(shape=(178, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)

    # -------------------------------------------------------------------------
    # Fuse (concatenate) the outputs of both branches
    # -------------------------------------------------------------------------
    merged = layers.Concatenate()([x1, x2])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    # Optionally, you can add more dense layers after concatenation
    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)

    # -------------------------------------------------------------------------
    # Build and compile the final model
    # -------------------------------------------------------------------------
    model = keras.Model(inputs=[input1, input2], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model


# -----------------------------------------------------------------------------
# Data Loading and Preprocessing
# -----------------------------------------------------------------------------
# 1) Load first input data (shape = (n_samples, 143, 4))
#    The following is analogous to your original code for the base + PAM input.
with open('Feature_CNN1.txt', 'r') as file:
    data_branch1 = []
    current_array = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array:
                data_branch1.append(current_array)
                current_array = []
        else:
            line = line.replace('[', '').replace(']', '')
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array:
        data_branch1.append(current_array)


X_branch1 = np.array(data_branch1)  # shape: (n_samples, 143, 4)
X_branch1 = X_branch1.reshape(len(X_branch1), 143, 4)  # (samples, timesteps, features)

# 2) Load second input data (shape = (n_samples, 178, 3))
#    Adjust this part to match how your second input file is formatted.
#    For example, if you have something like "Feature_branch2.txt" with dimension (84,3).
with open('Feature_CNN2.txt', 'r') as file:
    data_branch2 = []
    current_array2 = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array2:
                data_branch2.append(current_array2)
                current_array2 = []

        else:
            line = line.replace('[', '').replace(']', '')
            current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array2:
        data_branch2.append(current_array2)

X_branch2 = np.array(data_branch2)  # shape: (n_samples, 178, 3)
X_branch2 = X_branch2.reshape(len(X_branch2), 178, 3)  # (samples, timesteps, features)

# 3) Load regression targets (y)
#    Make sure this matches the number of samples for both inputs.
rate = []
with open('HT3_indel_frequency_value_percentage.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)

# -----------------------------------------------------------------------------
# Ensure both X_branch1 and X_branch2 have the same first dimension as y
# -----------------------------------------------------------------------------
assert len(X_branch1) == len(X_branch2) == len(y), "All inputs must have the same number of samples."


# -----------------------------------------------------------------------------
# (Optional) Randomly select or filter data if you want
# -----------------------------------------------------------------------------
# For illustration, let's assume you want to select the same 1000 random samples
# from both X_branch1 and X_branch2. Then create a hold-out set from the rest.
np.random.seed(42)
indices_to_select = np.random.choice(len(X_branch1), size=1000, replace=False)
all_indices = np.arange(len(X_branch1))
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

X1_selected = X_branch1[indices_to_select]
X2_selected = X_branch2[indices_to_select]
y_selected = y[indices_to_select]

X1_unselected = X_branch1[indices_unselected]
X2_unselected = X_branch2[indices_unselected]
y_unselected = y[indices_unselected]


# -----------------------------------------------------------------------------
# Split the selected data into training and testing sets
# -----------------------------------------------------------------------------
X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X1_selected, 
    X2_selected, 
    y_selected,
    test_size=0.2,
    random_state=42
)


# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1_CNN2.h5')

# -----------------------------------------------------------------------------
# Retrieve the Best Model and Evaluate
# -----------------------------------------------------------------------------
val_loss, val_mae = final_model.evaluate([X1_test, X2_test], y_test)
print(f"Best Fusion CNN model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict and compute Spearman correlation on the test set
y_pred_test = final_model.predict([X1_test, X2_test])
spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
print(f"Spearman Correlation (test set): {spearman_corr_test}")

print()


# -----------------------------------------------------------------------------
# Evaluate on new (unselected) data multiple times if desired
# -----------------------------------------------------------------------------
spearman_array = []

with open("outlier_feature_index_MLP.txt", "r") as f:
    outlier_feature_index = [int(line.strip()) for line in f]

# Randomly select 1000 samples from the unselected portion
indices_to_select_new = np.random.choice(outlier_feature_index, size=len(outlier_feature_index), replace=False)

X1_unselected_selected = X_branch1[indices_to_select_new]
X2_unselected_selected = X_branch2[indices_to_select_new]
y_unselected_selected = y[indices_to_select_new]


# Predict on the new data
y_pred_new = final_model.predict([X1_unselected_selected, X2_unselected_selected])
for i in range (0, len(y_pred_new)):
    print(i)
    print(y_pred_new[i][0])
    print(y_unselected_selected[i])
    print(abs(y_pred_new[i][0]-y_unselected_selected[i]))
    print()
    
with open("12_abs_error_CNN1_CNN2_MLP_not_included.txt", "w") as f:
    for i in range(len(y_pred_new)):
        abs_err = abs(y_pred_new[i][0] - y_unselected_selected[i])
        f.write(f"{abs_err}\n")

with open("12_predicted_CNN1_CNN2_MLP_not_included.txt", "w") as f:
    for i in range(len(y_pred_new)):
        f.write(f"{y_pred_new[i][0]}\n")

with open("12_actual_CNN1_CNN2_MLP_not_included.txt", "w") as f:
    for i in range(len(y_unselected_selected)):
        f.write(f"{y_unselected_selected[i]}\n")

y_pred_new = np.asarray(y_pred_new, dtype=np.float64).ravel()

# Compute Spearman correlation
pearson_corr_new, _ = pearsonr(y_unselected_selected, y_pred_new)
print(pearson_corr_new)



In [ ]:
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping


# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')
                

def build_fusion_model(hp):
    """
    Builds a three-branch model:
      - Branch 1: CNN for input shape (143, 4)
      - Branch 2: CNN for input shape (178, 3)
      - Branch 3: MLP for input shape (17,)
    The outputs from these branches are concatenated (fused) 
    and then passed through final Dense layers for regression.
    """

    # -------------------------------------------------------------------------
    # Branch 1 (CNN) for input of shape (143, 4)
    # -------------------------------------------------------------------------
    input1 = keras.Input(shape=(143, 4), name='branch1_input')

    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)


    # -------------------------------------------------------------------------
    # Branch 2 (CNN) for input of shape (178, 3)
    # -------------------------------------------------------------------------
    input2 = keras.Input(shape=(178, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)


    # -------------------------------------------------------------------------
    # Branch 3 (MLP) for input of shape (17,)
    # -------------------------------------------------------------------------
    input3 = keras.Input(shape=(17,), name='branch3_input')

    x3 = layers.Dense(
        hp.Int('dense_units3', min_value=16, max_value=256, step=16),
        activation='relu'
    )(input3)
    # Optionally add more layers in the MLP if desired, e.g.:
    # x3 = layers.Dense(hp.Int('dense_units3b', ...), activation='relu')(x3)

    # -------------------------------------------------------------------------
    # Fuse (concatenate) the outputs of all three branches
    # -------------------------------------------------------------------------
    merged = layers.Concatenate()([x1, x2, x3])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    # Optionally, you can add more Dense layers after concatenation
    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)


    # -------------------------------------------------------------------------
    # Build and compile the final model
    # -------------------------------------------------------------------------
    model = keras.Model(inputs=[input1, input2, input3], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model
    
    
def main():

    # -----------------------------------------------------------------------------
    # Data Loading and Preprocessing
    # -----------------------------------------------------------------------------
    # 1) Load Branch 1 data: shape (n_samples, 143, 4)
    with open('Feature_CNN1.txt', 'r') as file:
        data_branch1 = []
        current_array = []
        for line in file:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[', '').replace(']', '')
                current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
        if current_array:
            data_branch1.append(current_array)

    X_branch1 = np.array(data_branch1)  # shape: (n_samples, 143, 4)
    X_branch1 = X_branch1.reshape(len(X_branch1), 143, 4)  # (samples, timesteps, features)
    print("X_branch1 shape:", X_branch1.shape)

    # 2) Load Branch 2 data: shape (n_samples, 178, 3)
    with open('Feature_CNN2.txt', 'r') as file:
        data_branch2 = []
        current_array2 = []
        for line in file:
            line = line.strip()
            if line == "":
                if current_array2:
                    data_branch2.append(current_array2)

                    current_array2 = []
            else:
                line = line.replace('[', '').replace(']', '')
                current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
        if current_array2:
            data_branch2.append(current_array2)

    X_branch2 = np.array(data_branch2)  # shape: (n_samples, 178, 3)
    X_branch2 = X_branch2.reshape(len(X_branch2), 178, 3)  # (samples, timesteps, features)
    print("X_branch2 shape:", X_branch2.shape)

    # 3) Load Branch 3 data (MLP) => shape (n_samples, 17)
    # Adjust how you read the file to match its exact format
    X_branch3 = []
    with open('Feature_MLP.txt', 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line (space- or comma-delimited)
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)

    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)  # (samples, timesteps, features)
    print("X_branch3 shape:", X_branch3.shape)


    # 4) Load regression targets (y)
    rate = []
    with open('HT3_indel_frequency_value_percentage.txt', 'r') as file:
        rate = [float(line.strip()) for line in file.readlines()]
    y = np.array(rate)
    print("y shape:", y.shape)

    # -----------------------------------------------------------------------------
    # Check shape consistency
    # -----------------------------------------------------------------------------
    assert len(X_branch1) == len(X_branch2) == len(X_branch3) == len(y), \
        "All inputs must have the same number of samples."

    # -----------------------------------------------------------------------------
    # (Optional) Randomly select 8507 samples (like your example)
    # -----------------------------------------------------------------------------
    np.random.seed(42)
    indices_to_select = np.random.choice(len(X_branch1), size=1000, replace=False)
    all_indices = np.arange(len(X_branch1))
    indices_unselected = np.setdiff1d(all_indices, indices_to_select)


    X1_selected = X_branch1[indices_to_select]
    X2_selected = X_branch2[indices_to_select]
    X3_selected = X_branch3[indices_to_select]
    y_selected  = y[indices_to_select]

    X1_unselected = X_branch1[indices_unselected]
    X2_unselected = X_branch2[indices_unselected]
    X3_unselected = X_branch3[indices_unselected]
    y_unselected  = y[indices_unselected]

    # -----------------------------------------------------------------------------
    # Split the selected data into training and testing sets
    # -----------------------------------------------------------------------------
    X1_train, X1_test, X2_train, X2_test, X3_train, X3_test, y_train, y_test = train_test_split(
        X1_selected, 
        X2_selected,
        X3_selected,
        y_selected,
        test_size=0.2,
        random_state=42
    )


    print("Train shapes:", X1_train.shape, X2_train.shape, X3_train.shape, y_train.shape)
    print("Test shapes: ", X1_test.shape,  X2_test.shape,  X3_test.shape,  y_test.shape)


    # Load the saved overall best model for final evaluation.
    final_model = tf.keras.models.load_model('best_CNN1_CNN2_MLP.h5')

    # -----------------------------------------------------------------------------
    # Retrieve the Best Model and Evaluate
    # -----------------------------------------------------------------------------
    val_loss, val_mae = final_model.evaluate([X1_test, X2_test, X3_test], y_test)
    print(f"\nBest Fusion CNN+MLP model - Test Loss: {val_loss}, Test MAE: {val_mae}")

    # Predict and compute Spearman correlation on the test set
    y_pred_test = final_model.predict([X1_test, X2_test, X3_test])
    spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
    print(f"Spearman Correlation (test set): {spearman_corr_test}\n")


    # -----------------------------------------------------------------------------
    # Evaluate on the unselected data multiple times (Optional)
    # -----------------------------------------------------------------------------
    spearman_array = []
    with open("outlier_feature_index_MLP.txt", "r") as f:
        outlier_feature_index = [int(line.strip()) for line in f]

    indices_to_select_new = np.random.choice(outlier_feature_index, size=len(outlier_feature_index), replace=False)


    X1_unselected_selected = X_branch1[indices_to_select_new]
    X2_unselected_selected = X_branch2[indices_to_select_new]
    X3_unselected_selected = X_branch3[indices_to_select_new]
    y_unselected_selected  = y[indices_to_select_new]

    
    # Predict on the new data
    y_pred_new = final_model.predict([X1_unselected_selected, 
                                                   X2_unselected_selected,
                                                   X3_unselected_selected])

    for i in range (0, len(y_pred_new)):
        print(i)
        print(y_pred_new[i][0])
        print(y_unselected_selected[i])
        print(abs(y_pred_new[i][0]-y_unselected_selected[i]))
        print()
    
    with open("12_abs_error_CNN1_CNN2_MLP_included.txt", "w") as f:
        for i in range(len(y_pred_new)):
            abs_err = abs(y_pred_new[i][0] - y_unselected_selected[i])
            f.write(f"{abs_err}\n")

    with open("12_predicted_CNN1_CNN2_MLP_included.txt", "w") as f:
        for i in range(len(y_pred_new)):
            f.write(f"{y_pred_new[i][0]}\n")

    y_pred_new = np.asarray(y_pred_new, dtype=np.float64).ravel()
    
    # Compute Spearman correlation
    pearson_corr_new, _ = pearsonr(y_unselected_selected, y_pred_new)
    print(pearson_corr_new)



if __name__ == "__main__":
    main()

In [ ]:
# Calculate difference between CNN1+CNN2 and CNN1+CNN2+MLP predicted

In [ ]:
file1 = "12_abs_error_CNN1_CNN2_MLP_not_included.txt"
file2 = "12_abs_error_CNN1_CNN2_MLP_included.txt"
output = "difference_12_abs_error_MLP_included.txt"

i = 0

with open(file1) as f1, open(file2) as f2, open(output, "w") as fout:
    for line1, line2 in zip(f1, f2):
        print(i)
        v1 = float(line1.strip())
        v2 = float(line2.strip())
        diff = v1 - v2
        print(diff)
        fout.write(f"{diff}\n")
        i += 1


In [ ]:
# CNN1+CNN2+MLP vs. CNN1+CNN2+MLP+GNN

In [ ]:
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from keras.callbacks import EarlyStopping


# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')
                

def build_fusion_model(hp):
    """
    Builds a three-branch model:
      - Branch 1: CNN for input shape (143, 4)
      - Branch 2: CNN for input shape (178, 3)
      - Branch 3: MLP for input shape (17,)
    The outputs from these branches are concatenated (fused) 
    and then passed through final Dense layers for regression.
    """

    # -------------------------------------------------------------------------
    # Branch 1 (CNN) for input of shape (143, 4)
    # -------------------------------------------------------------------------
    input1 = keras.Input(shape=(143, 4), name='branch1_input')

    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)


    # -------------------------------------------------------------------------
    # Branch 2 (CNN) for input of shape (178, 3)
    # -------------------------------------------------------------------------
    input2 = keras.Input(shape=(178, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)


    # -------------------------------------------------------------------------
    # Branch 3 (MLP) for input of shape (17,)
    # -------------------------------------------------------------------------
    input3 = keras.Input(shape=(17,), name='branch3_input')

    x3 = layers.Dense(
        hp.Int('dense_units3', min_value=16, max_value=256, step=16),
        activation='relu'
    )(input3)
    # Optionally add more layers in the MLP if desired, e.g.:
    # x3 = layers.Dense(hp.Int('dense_units3b', ...), activation='relu')(x3)

    # -------------------------------------------------------------------------
    # Fuse (concatenate) the outputs of all three branches
    # -------------------------------------------------------------------------
    merged = layers.Concatenate()([x1, x2, x3])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    # Optionally, you can add more Dense layers after concatenation
    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)


    # -------------------------------------------------------------------------
    # Build and compile the final model
    # -------------------------------------------------------------------------
    model = keras.Model(inputs=[input1, input2, input3], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model
    
    
def main():

    # -----------------------------------------------------------------------------
    # Data Loading and Preprocessing
    # -----------------------------------------------------------------------------
    # 1) Load Branch 1 data: shape (n_samples, 143, 4)
    with open('Feature_CNN1.txt', 'r') as file:
        data_branch1 = []
        current_array = []
        for line in file:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[', '').replace(']', '')
                current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
        if current_array:
            data_branch1.append(current_array)

    X_branch1 = np.array(data_branch1)  # shape: (n_samples, 143, 4)
    X_branch1 = X_branch1.reshape(len(X_branch1), 143, 4)  # (samples, timesteps, features)
    print("X_branch1 shape:", X_branch1.shape)

    # 2) Load Branch 2 data: shape (n_samples, 178, 3)
    with open('Feature_CNN2.txt', 'r') as file:
        data_branch2 = []
        current_array2 = []
        for line in file:
            line = line.strip()
            if line == "":
                if current_array2:
                    data_branch2.append(current_array2)

                    current_array2 = []
            else:
                line = line.replace('[', '').replace(']', '')
                current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
        if current_array2:
            data_branch2.append(current_array2)

    X_branch2 = np.array(data_branch2)  # shape: (n_samples, 178, 3)
    X_branch2 = X_branch2.reshape(len(X_branch2), 178, 3)  # (samples, timesteps, features)
    print("X_branch2 shape:", X_branch2.shape)

    # 3) Load Branch 3 data (MLP) => shape (n_samples, 17)
    # Adjust how you read the file to match its exact format
    X_branch3 = []
    with open('Feature_MLP.txt', 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line (space- or comma-delimited)
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)

    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)  # (samples, timesteps, features)
    print("X_branch3 shape:", X_branch3.shape)


    # 4) Load regression targets (y)
    rate = []
    with open('HT3_indel_frequency_value_percentage.txt', 'r') as file:
        rate = [float(line.strip()) for line in file.readlines()]
    y = np.array(rate)
    print("y shape:", y.shape)

    # -----------------------------------------------------------------------------
    # Check shape consistency
    # -----------------------------------------------------------------------------
    assert len(X_branch1) == len(X_branch2) == len(X_branch3) == len(y), \
        "All inputs must have the same number of samples."

    # -----------------------------------------------------------------------------
    # (Optional) Randomly select 8507 samples (like your example)
    # -----------------------------------------------------------------------------
    np.random.seed(42)
    indices_to_select = np.random.choice(len(X_branch1), size=1000, replace=False)
    all_indices = np.arange(len(X_branch1))
    indices_unselected = np.setdiff1d(all_indices, indices_to_select)


    X1_selected = X_branch1[indices_to_select]
    X2_selected = X_branch2[indices_to_select]
    X3_selected = X_branch3[indices_to_select]
    y_selected  = y[indices_to_select]

    X1_unselected = X_branch1[indices_unselected]
    X2_unselected = X_branch2[indices_unselected]
    X3_unselected = X_branch3[indices_unselected]
    y_unselected  = y[indices_unselected]

    # -----------------------------------------------------------------------------
    # Split the selected data into training and testing sets
    # -----------------------------------------------------------------------------
    X1_train, X1_test, X2_train, X2_test, X3_train, X3_test, y_train, y_test = train_test_split(
        X1_selected, 
        X2_selected,
        X3_selected,
        y_selected,
        test_size=0.2,
        random_state=42
    )


    print("Train shapes:", X1_train.shape, X2_train.shape, X3_train.shape, y_train.shape)
    print("Test shapes: ", X1_test.shape,  X2_test.shape,  X3_test.shape,  y_test.shape)


    # Load the saved overall best model for final evaluation.
    final_model = tf.keras.models.load_model('best_CNN1_CNN2_MLP.h5')

    # -----------------------------------------------------------------------------
    # Retrieve the Best Model and Evaluate
    # -----------------------------------------------------------------------------
    val_loss, val_mae = final_model.evaluate([X1_test, X2_test, X3_test], y_test)
    print(f"\nBest Fusion CNN+MLP model - Test Loss: {val_loss}, Test MAE: {val_mae}")

    # Predict and compute Spearman correlation on the test set
    y_pred_test = final_model.predict([X1_test, X2_test, X3_test])
    spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
    print(f"Spearman Correlation (test set): {spearman_corr_test}\n")


    # -----------------------------------------------------------------------------
    # Evaluate on the unselected data multiple times (Optional)
    # -----------------------------------------------------------------------------
    spearman_array = []
    with open("outlier_feature_index_GNN.txt", "r") as f:
        outlier_feature_index = [int(line.strip()) for line in f]

    indices_to_select_new = np.random.choice(outlier_feature_index, size=len(outlier_feature_index), replace=False)


    X1_unselected_selected = X_branch1[indices_to_select_new]
    X2_unselected_selected = X_branch2[indices_to_select_new]
    X3_unselected_selected = X_branch3[indices_to_select_new]
    y_unselected_selected  = y[indices_to_select_new]

    
    # Predict on the new data
    y_pred_new = final_model.predict([X1_unselected_selected, 
                                                   X2_unselected_selected,
                                                   X3_unselected_selected])

    for i in range (0, len(y_pred_new)):
        print(i)
        print(y_pred_new[i][0])
        print(y_unselected_selected[i])
        print(abs(y_pred_new[i][0]-y_unselected_selected[i]))
        print()
    
    with open("12_abs_error_CNN1_CNN2_MLP_GNN_not_included.txt", "w") as f:
        for i in range(len(y_pred_new)):
            abs_err = abs(y_pred_new[i][0] - y_unselected_selected[i])
            f.write(f"{abs_err}\n")

    with open("12_predicted_CNN1_CNN2_MLP_GNN_not_included.txt", "w") as f:
        for i in range(len(y_pred_new)):
            f.write(f"{y_pred_new[i][0]}\n")

    with open("12_actual_CNN1_CNN2_MLP_GNN_not_included.txt", "w") as f:
        for i in range(len(y_unselected_selected)):
            f.write(f"{y_unselected_selected[i]}\n")

    y_pred_new = np.asarray(y_pred_new, dtype=np.float64).ravel()
    
    # Compute Spearman correlation
    pearson_corr_new, _ = pearsonr(y_unselected_selected, y_pred_new)
    print(pearson_corr_new)



if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset


# PyG
from torch_geometric.data import Data, Batch

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data():
    """
    Loads (143,4) data from "Feature_CNN1.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 143, 4)
    return X_branch1


def load_branch2_data():
    """
    Loads (178,3) data from "Feature_CNN2.txt".
    """
    data_branch2 = []
    current_array = []
    with open("Feature_CNN2.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 178, 3)
    return X_branch2


def load_branch3_data(filename="Feature_MLP.txt"):
    """
    Loads 17-dimensional numeric features from file.
    Each line has 17 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)
    return X_branch3


def load_reaction_rates():
    with open('HT3_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


########################################
# 2. Graph Data Utilities (for GNN)
########################################

def nucleotide_to_one_hot(nucleotide):
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'U': [0, 0, 0, 1],
        '-': [0, 0, 0, 0]
    }
    return mapping.get(nucleotide, [0,0,0,0])

def encode_sequence(sequence):
    return [nucleotide_to_one_hot(nuc) for nuc in sequence]

def matrix_to_edge_index(prob_matrix):
    edges_source = []
    edges_target = []
    L = len(prob_matrix)
    for i in range(L):
        for j in range(i + 1, L):  # only i < j
            if prob_matrix[i][j] > 0.01:
                edges_source.append(i)
                edges_target.append(j)
    return [edges_source, edges_target]

def generate_edge_features(edge_index, seq_one_hot, prob_matrix):
    features = []
    if not edge_index:
        return []
    for i in range(len(edge_index[0])):
        src_idx = edge_index[0][i]
        tgt_idx = edge_index[1][i]
        prob_val = prob_matrix[src_idx][tgt_idx]
        
        def decode_nt(one_hot_vec):
            if one_hot_vec[0] == 1:
                return 'A'
            elif one_hot_vec[1] == 1:
                return 'C'
            elif one_hot_vec[2] == 1:
                return 'G'
            elif one_hot_vec[3] == 1:
                return 'T'
            return 'N'
        
        src_char = decode_nt(seq_one_hot[src_idx])
        tgt_char = decode_nt(seq_one_hot[tgt_idx])
        
        if (src_char == 'A' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'A'):
            pair_type = 2
        elif (src_char == 'G' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'G'):
            pair_type = 2
        elif (src_char == 'C' and tgt_char == 'G') or (src_char == 'G' and tgt_char == 'C'):
            pair_type = 3
        
        features.append([pair_type/3, prob_val])
    return features

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def load_dna_reaction_data():
    with open('HT3_full_guide_sequences.txt', 'r') as f:
        gRNA_sequences = [line.strip() for line in f.readlines()]
    with open('HT3_target_sequences_noPAM.txt', 'r') as f:
        target_sequences = [line.strip() for line in f.readlines()]
    
    gRNA_node_features = [encode_sequence(seq) for seq in gRNA_sequences]
    
    target_bh_sequences = [target_sequences[i] + DNA_reverse_complement(target_sequences[i]) for i in range(len(target_sequences))]
    target_bh_node_features = [encode_sequence(seq) for seq in target_bh_sequences]
    
    duplex_sequences = [gRNA_sequences[i] + target_sequences[i] for i in range(len(target_sequences))]
    duplex_node_features = [encode_sequence(seq) for seq in duplex_sequences]

    target_ah_node_features = [encode_sequence(DNA_reverse_complement(seq)) for seq in target_sequences]

    gRNA_edge_indices = []
    gRNA_edge_features = []
    for guide in gRNA_sequences:
        prob_matrix = pairs(strands=guide, model=my_model_RNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in guide]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        gRNA_edge_indices.append(edge_index)
        gRNA_edge_features.append(feats)

    target_bh_edge_indices = []
    target_bh_edge_features = []
    for i in range (0, len(gRNA_sequences)):
        prob_matrix = pairs(strands=[target_sequences[i], DNA_reverse_complement(target_sequences[i])], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in target_bh_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        target_bh_edge_indices.append(edge_index)
        target_bh_edge_features.append(feats)
    
    duplex_edge_indices = []
    duplex_edge_features = []
    for i in range (0, len(gRNA_sequences)):
        prob_matrix = pairs(strands=[gRNA_sequences[i], target_sequences[i]], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in duplex_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        duplex_edge_indices.append(edge_index)
        duplex_edge_features.append(feats)

    target_ah_edge_indices = []
    target_ah_edge_features = [] 
    for target in target_sequences:
        prob_matrix = pairs(strands=DNA_reverse_complement(target), model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in DNA_reverse_complement(target)]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        target_ah_edge_indices.append(edge_index)
        target_ah_edge_features.append(feats)
    
    with open('HT3_indel_frequency_value_percentage.txt', 'r') as f:
        reaction_rates = [float(line.strip()) for line in f.readlines()]
    
    return (gRNA_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
            gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
            gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
            reaction_rates)

class GNNDataset(Dataset):
    def __init__(self, gRNA_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
                 gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
                 gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
                 reaction_rates):
        super().__init__()
        self.gRNA_node_features = gRNA_node_features
        self.target_bh_node_features = target_bh_node_features
        self.duplex_node_features = duplex_node_features
        self.target_ah_node_features = target_ah_node_features
        self.gRNA_edge_indices = gRNA_edge_indices
        self.target_bh_edge_indices = target_bh_edge_indices
        self.duplex_edge_indices = duplex_edge_indices
        self.target_ah_edge_indices = target_ah_edge_indices
        self.gRNA_edge_features = gRNA_edge_features
        self.target_bh_edge_features = target_bh_edge_features
        self.duplex_edge_features = duplex_edge_features
        self.target_ah_edge_features = target_ah_edge_features
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        c_x = torch.tensor(self.gRNA_node_features[idx], dtype=torch.float)
        c_edge_index_list = self.gRNA_edge_indices[idx]
        if not c_edge_index_list:
            c_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            c_edge_index = torch.tensor(c_edge_index_list, dtype=torch.long)
        c_edge_attr = torch.tensor(self.gRNA_edge_features[idx], dtype=torch.float)
        if c_edge_attr.dim() == 1:
            c_edge_attr = c_edge_attr.unsqueeze(-1).repeat(1,2)
        gRNA_data = Data(x=c_x, edge_index=c_edge_index, edge_attr=c_edge_attr)

        t_bh_x = torch.tensor(self.target_bh_node_features[idx], dtype=torch.float)
        t_bh_edge_index_list = self.target_bh_edge_indices[idx]
        if not t_bh_edge_index_list:
            t_bh_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            t_bh_edge_index = torch.tensor(t_bh_edge_index_list, dtype=torch.long)
        t_bh_edge_attr = torch.tensor(self.target_bh_edge_features[idx], dtype=torch.float)
        if t_bh_edge_attr.dim() == 1:
            t_bh_edge_attr = t_bh_edge_attr.unsqueeze(-1).repeat(1,2)
        target_bh_data = Data(x=t_bh_x, edge_index=t_bh_edge_index, edge_attr=t_bh_edge_attr)
        
        d_x = torch.tensor(self.duplex_node_features[idx], dtype=torch.float)
        d_edge_index_list = self.duplex_edge_indices[idx]
        if not d_edge_index_list:
            d_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            d_edge_index = torch.tensor(d_edge_index_list, dtype=torch.long)
        d_edge_attr = torch.tensor(self.duplex_edge_features[idx], dtype=torch.float)
        if d_edge_attr.dim() == 1:
            d_edge_attr = d_edge_attr.unsqueeze(-1).repeat(1,2)
        duplex_data = Data(x=d_x, edge_index=d_edge_index, edge_attr=d_edge_attr)

        t_ah_x = torch.tensor(self.target_ah_node_features[idx], dtype=torch.float)
        t_ah_edge_index_list = self.target_ah_edge_indices[idx]
        if not t_ah_edge_index_list:
            t_ah_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            t_ah_edge_index = torch.tensor(t_ah_edge_index_list, dtype=torch.long)
        t_ah_edge_attr = torch.tensor(self.target_ah_edge_features[idx], dtype=torch.float)
        if t_ah_edge_attr.dim() == 1:
            t_ah_edge_attr = t_ah_edge_attr.unsqueeze(-1).repeat(1,2)
        target_ah_data = Data(x=t_ah_x, edge_index=t_ah_edge_index, edge_attr=t_ah_edge_attr)
        
        y = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return gRNA_data, target_bh_data, duplex_data, target_ah_data, y

def collate(batch):
    from torch_geometric.data import Batch
    gRNA_list, target_bh_list, duplex_list, target_ah_list, y_list = zip(*batch)
    batch_gRNA = Batch.from_data_list(gRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_gRNA, batch_target_bh, batch_duplex, batch_target_ah, y

########################################
# 3. CNN, GNN, MLP Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (143,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*71, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,71)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,71)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

class CNNBranch2(nn.Module):
    """
    CNN branch for (178,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*89, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,178)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,89)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


class GNNBranch(nn.Module):
    def __init__(self, hidden_dim=32, edge_attr_dim=2):
        super().__init__()
        self.hidden_dim = hidden_dim

        def make_edge_mlp(edge_attr_dim, in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(edge_attr_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, in_channels * out_channels)
            )


        # gRNA
        self.edge_mlp_c = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_c2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_c = NNConv(4, hidden_dim, self.edge_mlp_c, aggr='mean')
        self.conv2_c = NNConv(hidden_dim, hidden_dim, self.edge_mlp_c2, aggr='mean')

        # target_bh
        self.edge_mlp_bh = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_bh2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_bh = NNConv(4, hidden_dim, self.edge_mlp_bh, aggr='mean')
        self.conv2_t_bh = NNConv(hidden_dim, hidden_dim, self.edge_mlp_bh2, aggr='mean')


        # duplex
        self.edge_mlp_d = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_d2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_d = NNConv(4, hidden_dim, self.edge_mlp_d, aggr='mean')
        self.conv2_d = NNConv(hidden_dim, hidden_dim, self.edge_mlp_d2, aggr='mean')

        # target_ah
        self.edge_mlp_ah = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_ah2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_ah = NNConv(4, hidden_dim, self.edge_mlp_ah, aggr='mean')
        self.conv2_t_ah = NNConv(hidden_dim, hidden_dim, self.edge_mlp_ah2, aggr='mean')


        # MLP after pooling all graph embeddings
        self.mlp_merge = nn.Sequential(
            nn.Linear(hidden_dim * 4 * 2, hidden_dim),  # 4 graphs, mean+max
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def pool(self, x, batch):
        return torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch)
        ], dim=1)


    def forward(self, gRNA_data, target_bh_data, duplex_data, target_ah_data):
        def process(graph_data, conv1, conv2):
            x, edge_index, edge_attr, batch = (
                graph_data.x, graph_data.edge_index, graph_data.edge_attr, graph_data.batch
            )
            x = F.elu(conv1(x, edge_index, edge_attr))
            x = F.elu(conv2(x, edge_index, edge_attr))
            return self.pool(x, batch)

        x_c = process(gRNA_data, self.conv1_c, self.conv2_c)
        x_t_bh = process(target_bh_data, self.conv1_t_bh, self.conv2_t_bh)
        x_d = process(duplex_data, self.conv1_d, self.conv2_d)
        x_t_ah = process(target_ah_data, self.conv1_t_ah, self.conv2_t_ah)

        merged = torch.cat([x_c, x_t_bh, x_d, x_t_ah], dim=1)
        return self.mlp_merge(merged)


class MLPBranch17(nn.Module):
    """
    A single hidden layer MLP for 17D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=17, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,17)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x

########################################
# 4. Final Fusion Model
########################################

class CNN_GNN_MLP_Fusion(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat_cnn1
      - CNNBranch2 => feat_cnn2
      - GNNBranch  => feat_gnn
      - MLPBranch17 => feat_mlp
    Concat => dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 filters2, kernel_size2, dense_units2,  # CNN2
                 gnn_hidden_dim,
                 mlp_hidden_dim,  # single hidden dimension for MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        self.gnn_branch = GNNBranch(hidden_dim=gnn_hidden_dim)
        self.mlp_branch = MLPBranch17(input_dim=17, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim)
        total_dim = dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    
    def forward(self, gRNA_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3):
        feat_cnn1 = self.cnn_branch1(x1)                   # (batch, dense_units1)
        feat_cnn2 = self.cnn_branch2(x2)                   # (batch, dense_units2)
        feat_gnn = self.gnn_branch(gRNA_data, target_bh_data, duplex_data, target_ah_data)  # (batch, gnn_hidden_dim)
        feat_mlp = self.mlp_branch(x3)                 # (batch, mlp_hidden_dim)
        
        merged = torch.cat([feat_cnn1, feat_cnn2, feat_gnn, feat_mlp], dim=1)
        # Apply dropout on the concatenated features
        merged = F.dropout(merged, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)


########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, graph_dataset, X1, X2, X3, reaction_rates):
        super().__init__()
        self.graph_dataset = graph_dataset
        self.X1 = X1
        self.X2 = X2
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(graph_dataset) == self.num_samples
        assert len(X1) == self.num_samples
        assert len(X2) == self.num_samples
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        gRNA_data, target_bh_data, duplex_data, target_ah_data, y_val = self.graph_dataset[idx]
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        x2 = torch.tensor(self.X2[idx], dtype=torch.float)  
        x3 = torch.tensor(self.X3[idx], dtype=torch.float)  
        return gRNA_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3, y_val

def hybrid_collate(batch):
    from torch_geometric.data import Batch
    gRNA_list, target_bh_list, duplex_list, target_ah_list, x1_list, x2_list, x3_list, y_list = zip(*batch)
    batch_gRNA = Batch.from_data_list(gRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    x1 = torch.stack(x1_list, dim=0)
    x2 = torch.stack(x2_list, dim=0)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_gRNA, batch_target_bh, batch_duplex, batch_target_ah, x1, x2, x3, y


from scipy.stats import pearsonr


########################################
# 6. Prediction only: GNN-included model
#    Evaluate ONLY indices listed in
#    outlier_feature_index_GNN.txt
########################################

MODEL_FILE = "best_FULL.pt"

ABS_ERROR_FILE = "12_abs_error_CNN1_CNN2_MLP_GNN_included.txt"
PREDICTED_FILE = "12_predicted_CNN1_CNN2_MLP_GNN_included.txt"

with open("outlier_feature_index_GNN.txt", "r") as f:
    outlier_feature_index = [int(line.strip()) for line in f]

np.random.seed(42)

X_branch1 = load_branch1_data()

indices_to_select = np.random.choice(
    len(X_branch1),
    size=1000,
    replace=False
)

# Then generate the outlier ordering
outlier_indices = np.random.choice(
    outlier_feature_index,
    size=len(outlier_feature_index),
    replace=False
)


def load_outlier_indices(filename, n_samples):
    """
    Read sample indices exactly in the order stored in the file.

    Indices are treated as 0-based, matching NumPy/PyTorch indexing and
    the behavior demonstrated in the comparison notebook.
    """
    indices = []

    with open(filename, "r") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                continue

            idx = int(line)

            if idx < 0 or idx >= n_samples:
                raise IndexError(
                    f"Index {idx} on line {line_number} of {filename} "
                    f"is outside the valid range 0 to {n_samples - 1}."
                )

            indices.append(idx)

    if len(indices) == 0:
        raise ValueError(f"No indices were found in {filename}.")

    return indices


def main_prediction():
    ########################################
    # Load all input features / labels
    ########################################
    X1 = load_branch1_data()
    X2 = load_branch2_data()
    X3 = load_branch3_data()
    y = load_reaction_rates()

    assert len(X1) == len(X2) == len(X3) == len(y), \
        "CNN1, CNN2, MLP, and label files must contain the same number of samples."

    print("Total number of samples:", len(y))

    ########################################
    # Generate the four GNN inputs
    ########################################
    (
        gRNA_node_feats,
        target_bh_node_feats,
        duplex_node_feats,
        target_ah_node_feats,
        gRNA_edge_indices,
        target_bh_edge_indices,
        duplex_edge_indices,
        target_ah_edge_indices,
        gRNA_edge_features,
        target_bh_edge_features,
        duplex_edge_features,
        target_ah_edge_features,
        _
    ) = load_dna_reaction_data()

    graph_dataset = GNNDataset(
        gRNA_node_feats,
        target_bh_node_feats,
        duplex_node_feats,
        target_ah_node_feats,
        gRNA_edge_indices,
        target_bh_edge_indices,
        duplex_edge_indices,
        target_ah_edge_indices,
        gRNA_edge_features,
        target_bh_edge_features,
        duplex_edge_features,
        target_ah_edge_features,
        y
    )

    hybrid_dataset = HybridDataset(
        graph_dataset,
        X1,
        X2,
        X3,
        y
    )

    ########################################
    # Read ONLY the requested unseen/outlier
    # sample indices
    ########################################

    print("Number of indices in outlier file:", len(outlier_indices))

    # The Subset preserves exactly the order in outlier_indices.
    unseen_set = Subset(
        hybrid_dataset,
        outlier_indices
    )

    unseen_loader = DataLoader(
        unseen_set,
        batch_size=1,
        shuffle=False,
        collate_fn=hybrid_collate
    )

    ########################################
    # Load the already-trained full model
    # NO TRAINING is performed here
    ########################################
    model = torch.load(
        MODEL_FILE,
        weights_only=False
    )
    model.eval()

    ########################################
    # Predict ONLY the indexed samples
    ########################################
    predicted_values = []
    actual_values = []

    with torch.no_grad():
        for (
            g_b,
            t_bh_b,
            d_b,
            t_ah_b,
            x1_b,
            x2_b,
            x3_b,
            y_b
        ) in unseen_loader:

            pred = model(
                g_b,
                t_bh_b,
                d_b,
                t_ah_b,
                x1_b,
                x2_b,
                x3_b
            )

            predicted_values.append(pred.item())
            actual_values.append(y_b.item())

    predicted_values = np.asarray(
        predicted_values
    )
    
    actual_values = np.asarray(
        actual_values
    )

    abs_errors = np.abs(
        predicted_values - actual_values
    )

    ########################################
    # Save requested output files
    ########################################
    with open(ABS_ERROR_FILE, "w") as f:
        for value in abs_errors:
            f.write(f"{value}\n")

    with open(PREDICTED_FILE, "w") as f:
        for value in predicted_values:
            f.write(f"{value}\n")

    ########################################
    # Pearson correlation on this unseen /
    # outlier-index dataset
    ########################################
    if len(actual_values) < 2:
        pearson_corr = np.nan
        print(
            "Pearson correlation cannot be computed "
            "with fewer than 2 samples."
        )
    elif np.std(actual_values) == 0 or np.std(predicted_values) == 0:
        pearson_corr = np.nan
        print(
            "Pearson correlation is undefined because "
            "actual or predicted values are constant."
        )
    else:
        pearson_corr, pearson_p = pearsonr(
            actual_values,
            predicted_values
        )

        print(
            "\nPearson correlation "
            "(actual vs predicted; outlier-index unseen dataset):",
            pearson_corr
        )
        print("Pearson p-value:", pearson_p)

    ########################################
    # Print line-by-line results
    ########################################

    for i in range (0, len(outlier_indices)):
        print(i)
        print(predicted_values[i])
        print(actual_values[i])
        print(abs_errors[i])
        print()

    print(pearson_corr)

    return {
        "indices": outlier_indices,
        "actual_values": actual_values,
        "predicted_values": predicted_values,
        "absolute_errors": abs_errors,
        "pearson_correlation": pearson_corr,
    }


if __name__ == "__main__":
    results = main_prediction()


In [ ]:
# Calculate difference between CNN1+CNN2+MLP and CNN1+CNN2+MLP_GNN predicted

In [ ]:
file1 = "12_abs_error_CNN1_CNN2_MLP_GNN_not_included.txt"
file2 = "12_abs_error_CNN1_CNN2_MLP_GNN_included.txt"
output = "difference_12_abs_error_GNN_included.txt"

i = 0

with open(file1) as f1, open(file2) as f2, open(output, "w") as fout:
    for line1, line2 in zip(f1, f2):
        print(i)
        v1 = float(line1.strip())
        v2 = float(line2.strip())
        diff = v1 - v2
        print(diff)
        fout.write(f"{diff}\n")
        i += 1
